# Bronze Ingestion — driver

This notebook contains **no ETL logic**. Everything lives in `src/bronze/` and `src/common/`
so it can be unit-tested, reviewed in a PR, and reused by a Job.

Per table: `ensure table → read watermark → capture ceiling T0 → pull [W−delay, T0]
→ add lineage → MERGE → advance watermark to T0`.

Prerequisite: run `sql/02_ddl_control.sql` once to create the `control` schema.

## 1. Make `src/` importable

When this repo is cloned as a Databricks **Git folder**, `src/` sits next to `notebooks/`.
Adjust the path if your folder differs.

In [0]:
import os
import sys


REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
SRC = os.path.join(REPO_ROOT, "src")

if SRC not in sys.path:
    sys.path.insert(0, SRC)

print("repo:", REPO_ROOT)
print("src on path:", SRC in sys.path)

## 2. Connection settings

Only the password is secret, and it is never written down — it is read from the
Databricks secret scope at run time !

In [0]:
from common.connections import SourceSettings, get_password

settings = SourceSettings(
    server="klitikalwa.database.windows.net",
    database="free-sql-db-9534149",
    user="adminzapas",
    secret_scope="ap_source",
    secret_key="azsql_password",
)

password = get_password(settings, dbutils)
print("connected as", settings.user, "->", settings.database)

## 3. Load config and sync it to `control.tables`

The YAML in Git is the source of truth; the control table is what the pipeline reads.
Syncing keeps them aligned and makes the active configuration queryable in SQL.

In [0]:
from common.config import load_configs, sync_config_to_control, get_enabled_table_configs

CONFIG_PATH = os.path.join(REPO_ROOT, "config", "ingestion_config.yaml")

file_configs = load_configs(CONFIG_PATH)
synced = sync_config_to_control(spark, file_configs)
print(f"synced {synced} table configs to control.tables")

configs = get_enabled_table_configs(spark)
print(f"{len(configs)} enabled tables")

for cfg in configs:
    print(f"  {cfg['source_table']:24s} {cfg['strategy']:12s} -> {cfg['target_table']}")

## 4. Run

Failures are isolated per table, recorded in `control.pipeline_runs`, then re-raised at
the end so a scheduled Job reports failure instead of a false green.

In [0]:
from bronze.pipeline import run_bronze

run_id = run_bronze(spark, settings, password, configs)
print("run_id:", run_id)

## 5. Verify

In [0]:
%sql
SELECT table_name, status, rows_read, rows_written, watermark_start, watermark_end,
       started_at, finished_at, error_message
FROM   workspace.control.pipeline_runs
ORDER  BY started_at DESC
LIMIT  40

In [0]:
%sql
SELECT table_name, watermark_column, last_watermark, last_successful_run_id, updated_at
FROM   workspace.control.watermarks
ORDER  BY table_name

### Reconcile source vs Bronze

Bronze is a mirror, so the counts should match. Any difference is a real gap worth chasing.

In [0]:
from common.connections import read_jdbc

for cfg in configs:
    source_rows = read_jdbc(spark, settings, password, cfg["source_table"]).count()
    target_rows = spark.table(cfg["target_table"]).count()
    flag = "ok" if source_rows == target_rows else "MISMATCH"
    print(f"{flag:9s} {cfg['source_table']:24s} source={source_rows:<8d} bronze={target_rows}")

### Idempotency proof

Re-run cell 4, then check the last write: `numTargetRowsInserted` should be **0** and the
row count unchanged. The overlap window re-reads a few rows; MERGE updates them to identical
values rather than duplicating them.

In [0]:
%sql
DESCRIBE HISTORY workspace.nb_bronze.invoice LIMIT 5